# Dataset Exporter: Convert Raw & Cleaned Data to CSV (`data_convert_to_csv.ipynb`)

This notebook loads the original RData dataset (`datasets/5v_cleandf.RData`) and exports:
1. **Raw (Uncleaned) Dataset**: All 972 columns and 560k+ observations saved to `datasets/5v_raw.csv`.
2. **Cleaned Dataset**: The 15 core features + ESI target with strict complete-cases filtering (zero null/NA values) based on `models/train_oof_logistic_regression_stacking.ipynb`, saved to `datasets/5v_cleaned.csv`.
3. **Stratified Splits**: The Train (98%), Validation (1%), and Holdout Test (1%) partitions saved to `datasets/train.csv`, `datasets/val.csv`, and `datasets/test.csv` based on `config/triage_conf.json`.

In [ ]:
# Load R magic extension for Python Jupyter kernel
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load 5v_cleandf.RData & Export Raw (Uncleaned) CSV
# ---------------------------------------------------------
data_file <- "datasets/5v_cleandf.RData"
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) data_file <- paste0("../", data_file)

out_dir <- if (dir.exists("datasets")) "datasets" else "../datasets"
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

cat(sprintf("Loading raw RData dataset from: %s\n", data_file))
data_env <- new.env()
load(data_file, envir = data_env)

df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
raw_df   <- get(df_names[which.max(df_sizes)], envir = data_env)

cat(sprintf("Raw Dataset Loaded: %d Rows x %d Columns\n", nrow(raw_df), ncol(raw_df)))

raw_csv_path <- file.path(out_dir, "5v_raw.csv")
cat(sprintf("Exporting Raw Dataset to %s ...\n", raw_csv_path))

write.csv(raw_df, file = raw_csv_path, row.names = FALSE)
cat(sprintf("✓ Successfully exported Raw Dataset to: %s (Size: %.2f MB)\n", 
            raw_csv_path, file.info(raw_csv_path)$size / (1024 * 1024)))


In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Build Cleaned Dataset Based on models/train_oof_logistic_regression_stacking.ipynb
# ---------------------------------------------------------
# 15 Core Raw Features + ESI Target
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(is.na(raw_df$gender), NA, ifelse(as.character(raw_df$gender) == "Male", 1, 0)) else rep(NA, nrow(raw_df))
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) raw_df$cc_breathingdifficulty else rep(NA, nrow(raw_df))

get_vec <- function(col_name) {
  if (col_name %in% names(raw_df)) return(raw_df[[col_name]]) else return(rep(NA, nrow(raw_df)))
}

target_col_name <- if ("esi" %in% names(raw_df)) "esi" else names(raw_df)[grep("esi", names(raw_df), ignore.case = TRUE)[1]]
raw_esi <- as.character(raw_df[[target_col_name]])

df_cleaned <- data.frame(
  age                     = raw_df$age,
  cc_breathingdifficulty  = cc_bd_vec,
  gender                  = gender_vec,
  triage_vital_hr         = get_vec("triage_vital_hr"),
  triage_vital_sbp        = get_vec("triage_vital_sbp"),
  triage_vital_rr         = get_vec("triage_vital_rr"),
  triage_vital_o2         = get_vec("triage_vital_o2"),
  pulse_min               = get_vec("pulse_min"),
  resp_min                = get_vec("resp_min"),
  spo2_min                = get_vec("spo2_min"),
  sbp_min                 = get_vec("sbp_min"),
  pulse_max               = get_vec("pulse_max"),
  resp_max                = get_vec("resp_max"),
  spo2_max                = get_vec("spo2_max"),
  sbp_max                 = get_vec("sbp_max"),
  esi                     = factor(raw_esi, levels = c("1", "2", "3", "4", "5"))
)

# Strictly drop any row containing at least 1 null/NA value across all 15 raw features or target
df_cleaned <- na.omit(df_cleaned)
df_cleaned$esi <- as.numeric(as.character(df_cleaned$esi))

clean_csv_path <- file.path(out_dir, "5v_cleaned.csv")
cat(sprintf("Exporting Cleaned Complete Cases Dataset (%d rows x %d cols) to %s ...\n", 
            nrow(df_cleaned), ncol(df_cleaned), clean_csv_path))

write.csv(df_cleaned, file = clean_csv_path, row.names = FALSE)
cat(sprintf("✓ Successfully exported Cleaned Dataset to: %s (Size: %.2f MB)\n", 
            clean_csv_path, file.info(clean_csv_path)$size / (1024 * 1024)))

cat("\nCleaned Dataset ESI Class Distribution:\n")
print(table(df_cleaned$esi))


In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Export Stratified Train / Validation / Test Splits (98% / 1% / 1%)
# ---------------------------------------------------------
stratified_sample <- function(y, fraction, seed = 42) {
  set.seed(seed)
  idx_list <- split(seq_along(y), y)
  sampled <- unlist(lapply(idx_list, function(idx) {
    n_sample <- max(1, round(length(idx) * fraction))
    sample(idx, size = n_sample)
  }))
  return(sort(sampled))
}

test_size <- 0.01
val_size  <- 0.01
seed_val  <- 42

# 1. Stratified Holdout Test Set (1%)
idx_test <- stratified_sample(df_cleaned$esi, test_size, seed = seed_val)
test_df  <- df_cleaned[idx_test, ]
rem_df   <- df_cleaned[-idx_test, ]

# 2. Stratified Validation Set (1% of total)
val_adj_frac <- val_size / (1 - test_size)
idx_val  <- stratified_sample(rem_df$esi, val_adj_frac, seed = seed_val + 1)
val_df   <- rem_df[idx_val, ]
train_df <- rem_df[-idx_val, ]

train_csv_path <- file.path(out_dir, "train.csv")
val_csv_path   <- file.path(out_dir, "val.csv")
test_csv_path  <- file.path(out_dir, "test.csv")

write.csv(train_df, file = train_csv_path, row.names = FALSE)
write.csv(val_df,   file = val_csv_path,   row.names = FALSE)
write.csv(test_df,  file = test_csv_path,  row.names = FALSE)

cat(sprintf("✓ Train Set Exported:      %s (%d rows, %.2f%%)\n", train_csv_path, nrow(train_df), (nrow(train_df)/nrow(df_cleaned))*100))
cat(sprintf("✓ Validation Set Exported: %s (%d rows, %.2f%%)\n", val_csv_path,   nrow(val_df),   (nrow(val_df)/nrow(df_cleaned))*100))
cat(sprintf("✓ Test Set Exported:       %s (%d rows, %.2f%%)\n", test_csv_path,  nrow(test_df),  (nrow(test_df)/nrow(df_cleaned))*100))


In [ ]:
import os
import pandas as pd

# ---------------------------------------------------------
# Step 4: Python Verification & Summary of Exported CSV Files
# ---------------------------------------------------------
target_dir = 'datasets' if os.path.exists('datasets') else '../datasets'
files_to_check = ['5v_raw.csv', '5v_cleaned.csv', 'train.csv', 'val.csv', 'test.csv']

summary_rows = []
for f in files_to_check:
    p = os.path.join(target_dir, f)
    if os.path.exists(p):
        sz_mb = os.path.getsize(p) / (1024 * 1024)
        # Read first 5 rows for column count and peek
        peek = pd.read_csv(p, nrows=5)
        summary_rows.append({
            'File': f,
            'Size (MB)': f"{sz_mb:.2f} MB",
            'Columns': len(peek.columns),
            'Sample Columns': ', '.join(peek.columns[:5]) + '...'
        })

summary_df = pd.DataFrame(summary_rows)
print("========================================================================================")
print("                      CSV EXPORT VERIFICATION SUMMARY")
print("========================================================================================")
print(summary_df.to_string(index=False))
print("========================================================================================")